# 05 Export: write the team data contract

Turns `merged.h5ad` into the files the web tool reads (same format as the synthetic data from
`src/make_fake_data.py`, so nothing downstream changes):

| file | contents |
|---|---|
| `cells.parquet` | id, x, y (UMAP), cluster (cell type), species, condition, donor, meld_score |
| `graph.npz` | symmetric 0/1 matrix: which cells are among each other's 15 nearest neighbours |
| `signaling.csv` | sender, receiver, ligand, receptor, score, condition, species |

`meld_score` is empty (NaN) until the MELD step fills it. `signaling.csv` is a **placeholder**:
for 10 well-known ligand–receptor pairs, score = mean ligand expression in the sender type ×
mean receptor expression in the receiver type. It is not CellPhoneDB (no permutation test).

Two exports: `export/full/` (all cells) and `export/n5000/` (5,000 cells, for the prototype).

In [1]:
import os, json, subprocess, sys
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
from config import *

adata = sc.read_h5ad(os.path.join(PROCESSED, "merged.h5ad"))
LR_PAIRS = [("SPP1", "CD44"), ("TGFB1", "TGFBR1"), ("PDGFB", "PDGFRB"), ("CCL2", "CCR2"), ("TNF", "TNFRSF1A"),
            ("IL1B", "IL1R1"), ("CXCL12", "CXCR4"), ("VEGFA", "KDR"), ("APOE", "TREM2"), ("CSF1", "CSF1R")]
LR_PAIRS = [(l, r) for l, r in LR_PAIRS if l in adata.var_names and r in adata.var_names]
print(adata.n_obs, "cells |", len(LR_PAIRS), "ligand-receptor pairs")

93077 cells | 10 ligand-receptor pairs


In [2]:
def cells_table(a):
    return pd.DataFrame({
        "id": np.arange(a.n_obs), "x": a.obsm["X_umap"][:, 0].astype(np.float32), "y": a.obsm["X_umap"][:, 1].astype(np.float32),
        "cluster": a.obs["cell_type"].astype(str).to_numpy(), "species": a.obs["species"].astype(str).to_numpy(),
        "condition": a.obs["condition"].astype(str).to_numpy(), "donor": a.obs["donor"].astype(str).to_numpy(),
        "meld_score": np.full(a.n_obs, np.nan, dtype=np.float32), "barcode": a.obs_names.to_numpy(),
    })

def graph(a):
    # Edges come from the kNN list itself (obsp["distances"]), not from UMAP connectivities:
    # UMAP weights for far neighbours can underflow to exactly 0, which silently drops kNN edges.
    g = a.obsp["distances"].tocsr().astype(np.float32, copy=True); g.data[:] = 1
    g = g.maximum(g.T).tocsr(); g.setdiag(0); g.eliminate_zeros()
    deg = np.diff(g.indptr)
    assert (g != g.T).nnz == 0, "graph must be symmetric"
    assert deg.min() >= N_NEIGHBORS - 1, f"a cell has only {deg.min()} neighbours"
    return g

def signaling(a):
    rows = []
    genes = sorted({g for p in LR_PAIRS for g in p})
    for (s, c), idx in a.obs.groupby(["species", "condition"], observed=True).groups.items():
        sub = a[idx, genes]
        m = pd.DataFrame(sub.X.toarray(), columns=genes).groupby(sub.obs["cell_type"].astype(str).to_numpy()).mean()
        for snd in m.index:
            for rcv in m.index:
                for l, r in LR_PAIRS:
                    rows.append((snd, rcv, l, r, round(float(m.at[snd, l] * m.at[rcv, r]), 4), c, s))
    return pd.DataFrame(rows, columns=["sender", "receiver", "ligand", "receptor", "score", "condition", "species"])

def write(a, out, **extra):
    os.makedirs(out, exist_ok=True)
    cells, g, sig = cells_table(a), graph(a), signaling(a)
    cells.to_parquet(os.path.join(out, "cells.parquet"), index=False)
    sp.save_npz(os.path.join(out, "graph.npz"), g)
    sig.to_csv(os.path.join(out, "signaling.csv"), index=False)
    json.dump({"source": "merged.h5ad", "n_cells": a.n_obs, "synthetic": False, "meld_score": "not yet computed",
               "signaling": "placeholder: mean ligand x mean receptor expression, not CellPhoneDB", **extra},
              open(os.path.join(out, "meta.json"), "w"), indent=2)
    print(f"{out}: {a.n_obs:,} cells, {g.nnz // 2:,} edges, {len(sig):,} signaling rows")

write(adata, os.path.join(EXPORT, "full"))

/Users/xinqing/Desktop/GT courses/Fall 2026/CSE6242 Data and Visual Analytics/Project/data/CSE6242_data/export/full: 93,077 cells, 1,149,880 edges, 3,620 signaling rows


## 5,000-cell subsample for the prototype
Equal numbers from each species × condition group, so both species show up. The kNN graph is
rebuilt on the subsample (in the same Harmony space); positions are kept from the full UMAP.

In [3]:
rng = np.random.default_rng(0)
grp = adata.obs.groupby(["species", "condition"], observed=True).indices
pick = np.sort(np.concatenate([rng.choice(ix, size=min(1250, len(ix)), replace=False) for ix in grp.values()]))
small = adata[pick].copy()
sc.pp.neighbors(small, n_neighbors=N_NEIGHBORS, use_rep="X_pca_harmony")
write(small, os.path.join(EXPORT, "n5000"), full_n_cells=adata.n_obs)

/opt/anaconda3/envs/scrnaseq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/xinqing/Desktop/GT courses/Fall 2026/CSE6242 Data and Visual Analytics/Project/data/CSE6242_data/export/n5000: 5,000 cells, 53,303 edges, 3,240 signaling rows


In [4]:
# Convert for the browser prototype (../prototype/data_real.json)
r = subprocess.run([sys.executable, "../src/export_for_web.py", os.path.join(EXPORT, "n5000"), "../prototype/data_real.json"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)

wrote ../prototype/data_real.json: 1.10 MB, 5000 cells, 24 cluster links
 
